# 恢复训练输入补齐：2019 OOF及旧开盘行集资格
固定原20日隔离间隔与300轮、三seed，仅扩展用于恢复训练所需2019输入；不修改2020–2022预测，不挑窗口。不将旧测试起点晚于新决策直接判为泄漏，实际比较训练行集及缩放器。

In [1]:
from pathlib import Path
from dataclasses import asdict
from io import BytesIO
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.workflow.minute_pit_allocator import _primary_dataset,_auxiliary_dataset,_prediction_dates,_ensemble
from src.workflow.stage1_open_action_value import ACTION_FEATURES
from src.model.tangent_allocator import TangentHeadAllocator,audit_tangent_initialization
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/tangent_recovery_inputs_2019_20261015';p=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013';os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
source,audit=_source(root,p['training_source'],['inputs/daily_frame','inputs/features','inputs/common','inputs/timing','inputs/native_prices','inputs/auxiliary','inputs/plan',*[f'models/base_2020_{seed}' for seed in p['base_seeds']],*[f'models/base_{year}_71' for year in [2021,2022]]])
frame=source['inputs/daily_frame'];calendar=pd.DatetimeIndex(source['inputs/native_prices'].xs('SOXX',level='instrument').index);purge=source['inputs/plan']['purge_rows']
for year in [2020,2021,2022]:assert calendar[calendar.searchsorted(pd.Timestamp(f'{year}-01-01'))-purge-1]==source[f'models/base_{year}_71'].train_end
train_end=calendar[calendar.searchsorted(pd.Timestamp('2019-01-01'))-purge-1];dates=frame.index[pd.to_datetime(frame.trade_date).dt.year.eq(2019)];assert len(dates)>200
original=source['models/base_2020_71'];inputs={'frame':frame,'features':source['inputs/features'],'common':source['inputs/common'],'timing':source['inputs/timing'],'spec':original.spec}
dataset=_primary_dataset(inputs,train_end=str(train_end.date()),predict_bounds=(str(dates.min().date()),str(dates.max().date())));auxiliary=_auxiliary_dataset(source['inputs/auxiliary']['2020'],str(train_end.date()));assert len(auxiliary.prepare('train'))==0
legacy=root/p['legacy_card'];manifest_path=legacy/'frozen_bundle_manifest.json';assert hashlib.sha256(manifest_path.read_bytes()).hexdigest()==p['legacy_manifest_sha256'];manifest=json.loads(manifest_path.read_text());meta=json.loads((legacy/'summary.json').read_text());assert manifest['recorder_id']==meta['recorder_id'];payloads={}
with R.uri_context(meta['recorder_uri']):
 rec_old=R.get_recorder(recorder_id=meta['recorder_id'],experiment_name=legacy.name)
 for name in ['action_value_training_frame.csv','open_action_value_models.pkl','training_diagnostics.csv','dataset_audit.json']:
  entry=next(e for e in manifest['artifacts'] if e['path'].endswith('/'+name));rel=entry['path'].split('/artifacts/',1)[1];data=Path(rec_old.download_artifact(rel)).read_bytes();assert hashlib.sha256(data).hexdigest()==entry['sha256'];payloads[name]=data
 models_old=rec_old.load_object(legacy.name+'/open_action_value_models.pkl')
legacy_train=pd.read_csv(BytesIO(payloads['action_value_training_frame.csv']));legacy_train['signal_date']=pd.to_datetime(legacy_train.signal_date);available=pd.to_datetime(legacy_train.label_available_at,utc=True);da=json.loads(payloads['dataset_audit.json']);diagnostics=pd.read_csv(BytesIO(payloads['training_diagnostics.csv']));eligibility=[]
for year in [2020,2021,2022]:
 name=f'train_{year}';segment=legacy_train.signal_date.between(*da['segments'][name]);old_cutoff=pd.Timestamp(da['training_cutoffs'][name]);first=pd.to_datetime(frame.loc[pd.to_datetime(frame.trade_date).dt.year.eq(year),'trade_date']).min();new_cutoff=first.tz_localize('America/New_York')+pd.Timedelta(hours=9,minutes=46)
 common=segment&available.notna()&legacy_train.execute_advantage.notna();a=common&available.lt(old_cutoff);b=common&available.lt(new_cutoff);assert a.equals(b)
 x=legacy_train.loc[b,list(ACTION_FEATURES)].to_numpy(float);model=models_old[year];imputer=model.named_steps['simpleimputer'];scaler=model.named_steps['standardscaler'];np.testing.assert_allclose(np.nanmean(x,axis=0),imputer.statistics_,rtol=1e-10,atol=1e-12);filled=np.where(np.isnan(x),imputer.statistics_,x);np.testing.assert_allclose(filled.mean(axis=0),scaler.mean_,rtol=1e-10,atol=1e-12);np.testing.assert_allclose(filled.var(axis=0),scaler.var_,rtol=1e-10,atol=1e-12)
 count=int(b.sum());assert count==int(diagnostics.loc[diagnostics.model_year.eq(str(year)),'training_rows'].iloc[0]);eligibility.append({'year':year,'rows':count,'old_cutoff':str(old_cutoff),'actual_first_decision':str(new_cutoff),'last_label_available_at':str(available[b].max()),'identical_training_row_set':True,'imputer_scaler_match':True,'actual_historical_model_publication_certified':False})
predictions={};training=[]
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,source_audit=audit,legacy_manifest=manifest,legacy_artifacts=payloads);rec.log_params(maximum_new_fits=3,new_backtests=0)
 for seed in p['base_seeds']:
  original=source[f'models/base_2020_{seed}'];spec={**asdict(original.spec),'root':str(root)};model=TangentHeadAllocator(base_features=original.base_features,extra_features=[],feature_masks=original.feature_masks,spec=spec,seed=seed+2019,cutoff='2019-01-01 00:00:00-05:00',train_end=str(train_end.date()),minimum_observations=original.minimum_observations)
  init=audit_tangent_initialization(model.__class__.__mro__[1]._initialize(model),model._initialize());model.fit(dataset,auxiliary_dataset=auxiliary);assert model.audit['epochs']==300 and model.audit['auxiliary_rows']==0;assert pd.Timestamp(model.audit['last_mature_label_at'])<pd.Timestamp('2019-01-01',tz='America/New_York')
  prediction=_prediction_dates(model.predict(dataset));assert prediction.index.equals(dates) and np.isfinite(prediction.to_numpy(float)).all();predictions[('2019',seed)]=prediction;training.append({'year':2019,'base_seed':seed,'actual_seed':model.seed,'prediction_rows':len(prediction),**model.audit,'initialization':init});rec.save_objects(artifact_path='models',**{f'2019_{seed}':model});rec.log_metrics(completed_fits=len(training))
 ensemble=_ensemble(predictions,'2019',p['base_seeds']);ensemble['trade_date']=frame.loc[ensemble.index,'trade_date'];ensemble['model_year']=2019;rec.save_objects(artifact_path='analysis',predictions=predictions,oof_predictions=ensemble,training=training,legacy_open_eligibility=pd.DataFrame(eligibility));rec.log_metrics(oof_rows=len(ensemble),registered_artifact_count=11)
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','analysis/predictions','analysis/oof_predictions','analysis/training']};summary={'experiment':card.name,'decision':'recovery_training_inputs_ready_not_promoted','new_fits':len(training),'new_backtests':0,'oof_rows':len(ensemble),'train_end':str(train_end.date()),'first_prediction_signal':str(dates.min().date()),'last_prediction_signal':str(dates.max().date()),'legacy_open_row_equivalence_passed':True,'annual_recovery_controller_ready':False,'provider_uri':str(provider),'region':'us','as_of':'2026-10-09','recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'objective_achieved':False,'scope':p['scope']}
pd.DataFrame(eligibility).to_csv(card/'legacy_open_eligibility.csv',index=False);pd.DataFrame([{k:v for k,v in row.items() if k not in ['initialization','observed_minute_training_rows','extra_scaler']} for row in training]).to_csv(card/'training_audit.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}));binding=json.loads((root/'research/tangent_router_oof_20261015/nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2));a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2))
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(pd.DataFrame(eligibility).to_string(index=False));print(summary['oof_rows'],summary['train_end'])


# 恢复训练输入补齐：2019 OOF与开盘旧模型行集资格

完成3次新的配置模型拟合，0新增账户回测。补齐2019年252个交易日的时间外配置预测，仅作为2020年前恢复训练的输入准备；既有2020–2022预测及评价窗口保持不变，不按收益选择年份。

沿用原39主输入、原目标与共同编码器、20行隔离、300轮固定末轮、三seed等权。先用同一Qlib原生价格日历复核2020/2021/2022已有训练截止一致，再推导2019训练截止为2018-11-29。每模型1992条成熟主样本、0辅助样本，最后标签成熟2018-12-10 09:31 ET，早于2019-01-01。缺乏更早合格辅助数据如实保留，不填入未来成分。实际seed为2090/2191/2292，按原base_seed+年度规则取得。

预测信号2018-12-31至2019-12-30，对应2019年252个实际交易日；模型、原始三seed预测、等权预测和初始化/成熟时间审计均保存Recorder。没有训练年度恢复分类器，也没有把2019评价收益用于参数选择。结构曾经过后续历史研究选择，故OOF仅表示拟合时序，不是新的独立策略验证。

旧年度开盘模型的首日资格也已核实。通过冻结bundle哈希读取实际训练帧、模型、训练诊断及Dataset审计，分别按旧cutoff和新首日09:46重建行集：

| 年度模型 | 两cutoff共同训练行数 | 最后标签成熟ET | 新首日决策ET |
|---|---:|---|---|
| 2020 | 1005 | 2020-01-02 09:31 | 2020-01-02 09:46 |
| 2021 | 1258 | 2021-01-04 09:31 | 2021-01-04 09:46 |
| 2022 | 1510 | 2022-01-03 09:31 | 2022-01-03 09:46 |

三组行集逐项相同；训练帧重算的imputer均值、标准化均值/方差与保存模型一致，行数与原诊断一致。原测试起点更晚并不能推出这些首日已有未来标签泄漏。此项只验证训练信息截止与预处理，不认证历史模型实际发布时间、原始行情首收、数据修订PIT或新价格/费用环境下的预测价值，也不补发完整策略资格。

年度恢复模型仍缺。下一步使用明确的2019训练输入与后续已成熟年度数据构造恢复动作标签，先检查样本、时点、费用和跨年状态连续性，再做固定年度训练及完整账户对照；不能将2023冻结恢复模型放回2020。新控制器与旧25bp标签开盘模型的费用/分布差异也须保留，不能仅凭可用行集相同宣称效果改善。

决策recovery_training_inputs_ready_not_promoted。生产未改，无新增收益证据，稳定净SOXX超额目标未完成。执行notebook与NAS冷恢复逐对象哈希通过。
